In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import TimeSeriesSplit

In [2]:
#Dynamic search path configuration
import sys
import os

#Adjusting search path based on notebook's depth in directory --> iterates through directories until it finds directory name: US_conflict_forecasting
current_dir = os.getcwd()
while 'US_conflict_forecasting' not in os.path.basename(current_dir):
    current_dir = os.path.dirname(current_dir)

#Adjusting search path based on 'new' current directory
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

from project_setup import setup_project
project_root_dir = setup_project()

import pandas as pd

Current Project Root Directory set to: /Users/danie/Desktop/Hertie/2. Semester/1. ML/0. Final Project/US_conflict_forecasting
Current Search Path set to: ['c:\\Users\\danie\\Desktop\\Hertie\\2. Semester\\1. ML\\0. Final Project\\US_conflict_forecasting', 'c:\\Users\\danie\\Desktop\\Hertie\\2. Semester\\1. ML\\0. Final Project\\US_conflict_forecasting\\3_Models', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\DLLs', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312\\Lib', 'c:\\Users\\danie\\AppData\\Local\\Programs\\Python\\Python312', '', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\win32\\lib', 'C:\\Users\\danie\\AppData\\Roaming\\Python\\Python312\\site-packages\\Pythonwin', 'c:\\Users\\danie\\AppData\

### Load the data

In [ ]:
# Load the data

data = pd.read_csv(os.path.join(project_root_dir, 'Dataset/acs_conflict_combined_cleaned.csv'))

#data = pd.read_csv(project_root_dir, 'acs_conflict_combined_cleaned.csv')

### Features and labels, cross-validation, split and normalization

In [ ]:
# Define features and label

X = data.drop(['event_count', 'ZCTA', 'Month'], axis=1)
Y = data['event_count']

In [ ]:
# Time Series Cross Validation, 5 folds.

tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
# Split the dataset into training and test sets
# 70/30 split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=1)

In [ ]:
# Normalize the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Initialize and fit the XGBoost Regressor

In [ ]:
xgb = XGBRegressor(use_label_encoder = False, eval_metric = 'logloss', random_state = 1 )
xgb.fit(X_train_scaled, Y_train)

### Run Grid Search to find the best parameters:

In [ ]:
param_grid = {
    'max_depth': [3, 4, 5, 6, 7],
    'min_child_weight': [1, 2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'n_estimators': [50, 100, 150, 200],
    'subsample': [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9]
}

In [ ]:
# Initialize XGB and GridSearch
grid_search = GridSearchCV(xgb, param_grid, scoring='accuracy', cv=5, verbose=1)
grid_search.fit(X_train_scaled, Y_train)

# Print the best parameters
best_params = grid_search.best_params_
print("Best parameters:", best_params)

### Re-run the model with optimal parameter values

In [ ]:
# Re-train the model with the best parameters found above
xgb_optimized = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state = 1)
xgb_optimized.fit(X_train_scaled, Y_train)

# Make predictions
y_pred_optimized = xgb_optimized.predict(X_test_scaled)

# Accurcary score
accuracy_optimized = accuracy_score(Y_test, y_pred_optimized)
print("Test set accuracy with optimized parameters: {:.2f}%".format(accuracy_optimized * 100))

### Run feature importance analysis

In [ ]:
importances = xgb.feature_importances_
indices = np.argsort(importances)[::-1]
plt.figure()
plt.title('Feature Importances')
plt.bar(range(X_train.shape[1]), importances[indices], color='r', align='center')
plt.xticks(range(X_train.shape[1]), X_train.columns[indices], rotation=90)
plt.xlim([-1, X_train.shape[1]])
plt.show()

### Find best number of features

In [ ]:
# Empty list to store and run different cv scores
cv_scores = []

for i in range(1, len(indices) + 1):
    # Select the top 'i' features
    top_features = [X_train.columns[indices[j]] for j in range(i)]
    
    # Create the training data using the selected features
    X_train_top = X_train_scaled[:, indices[:i]]
    
    # Initialize the model with the best parameters
    model = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
    
    # Perform cv at 5 and store the mean score
    scores = cross_val_score(model, X_train_top, Y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

# Which number of features gives the highest cv score?
optimal_features = np.argmax(cv_scores) + 1 
optimal_score = cv_scores[optimal_features - 1]

print(f"Optimal number of features: {optimal_features}")
print(f"Highest CV accuracy: {optimal_score:.2f}%")

### Plot

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(indices) + 1), cv_scores, marker='o')
plt.xlabel('Number of Features')
plt.ylabel('CV Accuracy')
plt.title('CV Performance vs. Number of Features')
plt.show()

### Re-run with best features

In [ ]:


# Select opimal features
top_features_indices = indices[:optimal_features]
X_train_optimal = X_train_scaled[:, top_features_indices]
X_test_optimal = X_test_scaled[:, top_features_indices]

# Train the model using the selected subset of features
model_optimal = XGBClassifier(**best_params, use_label_encoder=False, eval_metric='logloss', random_state=42)
model_optimal.fit(X_train_optimal, y_train)

# Evaluate the model on the test set
y_pred_optimal = model_optimal.predict(X_test_optimal)
accuracy_optimal = accuracy_score(y_test, y_pred_optimal)

print(f"Test set accuracy with optimal number of features ({optimal_features}): {accuracy_optimal:.2f}%")